# v4_deepsets — entrenamiento en Google Colab

Wrapper para correr `python/train_v4.py` en Colab (GPU T4/A100 gratis o Pro).

## Setup previo (una sola vez)
1. En Google Drive, subí toda la carpeta `PMDKernel/` a `MyDrive/Ipre/PMDKernel/`.
2. Tiene que incluir `python/` y `data/datasets/<tu_dataset>.h5`.
3. En Colab: **Runtime → Change runtime type → GPU**.

## Estimaciones
- Dataset n=500 (smoke): 3 epochs ~1 min en T4.
- Dataset n=5000, 100 epochs: ~15–30 min en T4.

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Instalar dependencias

Colab ya trae `torch` con CUDA. Solo falta agregar el resto.

In [ ]:
!pip install -q pytorch-lightning comet-ml h5py torchmetrics

In [ ]:
import torch, pytorch_lightning as pl, comet_ml, h5py, sklearn, torchmetrics
print(f"torch        = {torch.__version__}  cuda={torch.cuda.is_available()}")
print(f"lightning    = {pl.__version__}")
print(f"comet_ml     = {comet_ml.__version__}")
print(f"h5py         = {h5py.__version__}")
print(f"torchmetrics = {torchmetrics.__version__}")
if torch.cuda.is_available():
    print(f"device       = {torch.cuda.get_device_name(0)}")

## 3. Montar Google Drive y entrar al proyecto

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT = '/content/drive/MyDrive/Ipre/PMDKernel'
assert os.path.isdir(PROJECT), f'No existe {PROJECT} — revisa la ruta en Drive.'
os.chdir(PROJECT)
print('cwd:', os.getcwd())
!ls

## 4. Credenciales Comet (opcional)

Si querés trackear en Comet ML, completá las dos variables y corré la celda. Sino, saltala y agregá `os.environ['COMET_OFFLINE'] = '1'` o desactivá el logger.

In [ ]:
import os
os.environ['COMET_API_KEY']   = ''  # tu API key de comet.com
os.environ['COMET_WORKSPACE'] = 'sebasti-n-vallejos'
os.environ['COMET_PROJECT_NAME'] = 'pmdkernel'

## 5. Entrenar

Editá `H5`, `EPOCHS`, `RUN_TAG` según corresponda. Para un smoke test rápido, dejá `--quick` (3 epochs).

In [ ]:
H5      = 'data/datasets/v1_xy100_z225_step10_n5000.h5'
RUN_TAG = 'v4_deepsets_colab_n5000'
EPOCHS  = 100

!python python/train_v4.py \
    --h5 {H5} \
    --run-tag {RUN_TAG} \
    --epochs {EPOCHS} \
    --patience 20 \
    --points-per-sample 4096 \
    --num-workers 2 \
    --pin-memory

### Smoke test (3 epochs)
Para verificar que el pipeline corre antes de tirarle 100 epochs.

In [ ]:
!python python/train_v4.py \
    --h5 data/datasets/v1_xy100_z225_step10_n500.h5 \
    --run-tag v4_deepsets_colab_smoke \
    --quick \
    --num-workers 2 \
    --pin-memory

## 6. Inspeccionar resultados

El script guarda checkpoint + aux en `python/Models/v4_deepsets/logs/<run_tag>/`. Como esa carpeta está dentro de Drive, queda persistido automáticamente.

In [ ]:
import torch
from pathlib import Path

log_dir = Path('python/Models/v4_deepsets/logs') / RUN_TAG
aux_path = log_dir / f'{RUN_TAG}_aux.pt'
aux = torch.load(aux_path, weights_only=False)

print('checkpoint:', aux['ckpt_path'])
print('comet url :', aux['comet_url'])
print()
for split, m in aux['metrics'].items():
    print(f"{split:5s}  rmse={m['rmse_mt']:.4f} mT  r2={m['r2']:.4f}")